# Running the benchmark with a custom sklearn-like model

### Step 1: Import libraries

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from fnirsatlas.classification import run_benchmark
from fnirsatlas.feature_extraction import WindowedFeatureExtractor

### Step 2: Define model pipeline

Create a 3-step pipeline:
1. **WindowedFeatureExtractor**: Extracts statistical features from non-overlapping temporal windows
2. **StandardScaler**: Z-normalizes features across samples
3. **LogisticRegression**: Linear classifier with $l_2$ regularization

In [2]:
def get_clf(**kwargs):
    random_state = kwargs.pop("random_state")
    return Pipeline(
        [
            ("extractor", WindowedFeatureExtractor()),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(random_state=random_state, max_iter=100000)),
        ]
    )

### Step 3: Run benchmark

Execute cross-subject classification with hyperparameter grid search using nested cross-validation (5×5-fold outer CV, 4-fold inner CV by default).

In [ ]:
results = run_benchmark(
    get_clf=get_clf,
    classification_setting="cross",  # or "within"
    suite="ma",
    seed=42,
    n_jobs=1,
    param_grid={
        "extractor__n_windows": [1, 6], # Grid search supports feature extraction as well
        "clf__C": [1e-3, 1, 1e3], # Regularization strength
    },
)